## 🧠 SQL-Based Business Analysis
This notebook uses SQL queries to extract key business metrics and insights from the dataset, simulating real-world analytical workflows used in production systems.

In [5]:
%pip install ipython-sql


   ---------------------------------------- 0/4 [ipython-genutils]
   ---------- ----------------------------- 1/4 [sqlparse]
   ---------- ----------------------------- 1/4 [sqlparse]
   ---------- ----------------------------- 1/4 [sqlparse]
   -------------------- ------------------- 2/4 [prettytable]
   ------------------------------ --------- 3/4 [ipython-sql]
   ---------------------------------------- 4/4 [ipython-sql]

Note: you may need to restart the kernel to use updated packages.


In [10]:
%load_ext sql
%sql sqlite:///../data/food_delivery.db
%config SqlMagic.autopandas = True
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


### 📥 Data Integration with SQL
Loaded the cleaned dataset into an SQL environment to perform structured queries and business analysis.

In [11]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../data/food_delivery.db")

orders = pd.read_csv("../data/processed/clean_orders.csv")

orders.to_sql("orders", conn, if_exists="replace", index=False)

21321

### 💰 Revenue Analysis
Used SQL aggregation functions to calculate total revenue and understand overall business performance.

In [13]:
%%sql
SELECT
    SUM(bill_subtotal) AS revenue_before_discount,
    SUM(total) AS revenue_after_discount,
    SUM(bill_subtotal - total) AS total_discount_given,
    ROUND(
        SUM(bill_subtotal - total) * 100.0 / SUM(bill_subtotal),
        2
    ) AS discount_percentage
FROM orders;

 * sqlite:///../data/food_delivery.db
Done.


,revenue_before_discount,revenue_after_discount,total_discount_given,discount_percentage
0,15992388.26,14554058.14,1438330.12,8.99


### 📦 Order & Customer Metrics
Analyzed order volume and customer activity using SQL queries to identify engagement patterns.

In [14]:
%%sql
SELECT
    order_status,
    COUNT(*) AS total_orders,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders),2) AS percentage
FROM orders
GROUP BY order_status;

 * sqlite:///../data/food_delivery.db
Done.


,order_status,total_orders,percentage
0,Delivered,21131,99.11
1,Picked up,3,0.01
2,Rejected,158,0.74
3,Return cancelled,3,0.01
4,Returned,25,0.12
5,Timed out,1,0.00


### 📊 KPI Computation
Calculated key performance indicators such as:
- Total Orders  
- Average Order Value  
- Revenue Trends  

These metrics provide a quick overview of business health.

In [15]:
%%sql
SELECT
    restaurant_name,
    COUNT(customer_complaint_tag) AS complaint_count
FROM orders
WHERE customer_complaint_tag IS NOT NULL
GROUP BY restaurant_name
ORDER BY complaint_count DESC
LIMIT 10;

 * sqlite:///../data/food_delivery.db
Done.


,restaurant_name,complaint_count
0,Aura Pizzas,355
1,Swaad,105
2,Dilli Burger Adda,8
3,The Chicken Junction,1


In [17]:
%%sql
SELECT
    COUNT(*) AS total_orders,
    SUM(CASE WHEN rider_wait_time > kpt_duration THEN 1 ELSE 0 END) AS rider_wait_cases,
    ROUND(
        SUM(CASE WHEN rider_wait_time > kpt_duration THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS rider_wait_percentage
FROM orders;

 * sqlite:///../data/food_delivery.db
Done.


,total_orders,rider_wait_cases,rider_wait_percentage
0,21321,196,0.92


In [19]:
%%sql
SELECT
    subzone,
    COUNT(order_id) AS total_orders,
    ROUND(SUM(total),2) AS revenue
FROM orders
GROUP BY subzone
ORDER BY revenue DESC;

 * sqlite:///../data/food_delivery.db
Done.


,subzone,total_orders,revenue
0,Greater Kailash 2 (GK2),7380,4756247.48
1,Sector 4,6530,4494094.64
2,DLF Phase 1,3686,2604429.60
3,Sector 135,2442,1759093.61
4,Vasant Kunj,920,689695.79
5,Shahdara,360,249106.40
6,Chittaranjan Park,2,949.62
7,Sikandarpur,1,441.00


### 💡 Key Insights

- 💰 **Revenue is driven by order volume and order value**
- 👥 **Customer activity varies significantly across users**
- 📊 **Aggregated metrics provide a clear view of business performance**

### 🧠 Business Recommendations

- 📈 Focus on increasing average order value through upselling  
- 🎯 Identify and target high-frequency customers  
- ⚡ Focus on increasing raiders in subzones with highest revenue